# Substitution Effect and Income Effect: Food and T-shirts

## Learning goal
This notebook shows how a change in the **price of Food** affects consumer choice through:
- the **substitution effect**
- the **income effect**

## Economic setup
We study a consumer choosing between two **normal goods**:
- **Food** on the **horizontal axis**
- **T-shirts** on the **vertical axis**

Preferences are represented by a **CES utility function**:

$$U(F,T)=\left[\alpha F^{\rho} + (1-\alpha)T^{\rho}\right]^{1/\rho}$$

where $F$ = quantity of Food, $T$ = quantity of T-shirts, $0 < \alpha < 1$, $0 < \rho < 1$.

With these preferences, both goods are treated as **normal goods**.

## Key points in the graph

| Point | Meaning |
|---|---|
| **A** | Initial optimum |
| **B** | Compensated optimum (same utility as A, new prices) |
| **C** | Final optimum |

- **A to B** = substitution effect
- **B to C** = income effect
- **A to C** = total effect

## How to use the sliders

**Example 1 — price increase:** Old pF = 2, New pF = 4. Food becomes more expensive: both effects push Food demand down.

**Example 2 — price decrease:** Old pF = 4, New pF = 2. Food becomes cheaper: both effects push Food demand up.

**Example 3 — stronger preference for Food:** set alpha = 0.65 and compare with alpha = 0.35.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def utility(F, T, alpha=0.5, rho=0.5):
    F = np.asarray(F)
    T = np.asarray(T)
    eps = 1e-8
    F = np.maximum(F, eps)
    T = np.maximum(T, eps)
    return (alpha * (F ** rho) + (1 - alpha) * (T ** rho)) ** (1 / rho)

def marshallian_bundle(m=300, pF=2, pT=1, alpha=0.5, rho=0.5, n_grid=4000):
    eps = 1e-6
    F_max = max(m / pF - eps, eps * 10)
    F_grid = np.linspace(eps, F_max, n_grid)
    T_grid = (m - pF * F_grid) / pT
    T_grid = np.maximum(T_grid, eps)
    U_grid = utility(F_grid, T_grid, alpha, rho)
    i = int(np.argmax(U_grid))
    return F_grid[i], T_grid[i]

def tshirts_for_target_utility(F, target_u, alpha=0.5, rho=0.5):
    eps = 1e-8
    F = np.maximum(F, eps)
    target_u = max(target_u, eps)
    inside = (target_u ** rho - alpha * (F ** rho)) / (1 - alpha)
    inside = np.maximum(inside, eps)
    return inside ** (1 / rho)

def hicksian_bundle(target_u, pF=4, pT=1, alpha=0.5, rho=0.5, F_upper=200, n_grid=4000):
    eps = 1e-6
    F_grid = np.linspace(eps, F_upper, n_grid)
    T_grid = tshirts_for_target_utility(F_grid, target_u, alpha, rho)
    expenditure = pF * F_grid + pT * T_grid
    i = int(np.argmin(expenditure))
    return F_grid[i], T_grid[i], expenditure[i]

def budget_line(F, m, pF, pT):
    return (m - pF * F) / pT

def auto_windows(m, pF0, pF1, pT, F_A, T_A, F_B, T_B, F_C, T_C):
    x_candidates = [m / pF0, m / pF1, F_A, F_B, F_C]
    y_candidates = [m / pT, T_A, T_B, T_C]
    x_max = 1.18 * max(x_candidates)
    y_max = 1.18 * max(y_candidates)
    return max(x_max, 10), max(y_max, 10)

def draw_horizontal_effect(ax, x0, x1, y, label, value):
    ax.plot([x0, x1], [y, y], linewidth=1.8, clip_on=False)
    ax.plot([x0, x0], [y - 0.015 * ax.get_ylim()[1], y + 0.015 * ax.get_ylim()[1]], linewidth=1.2, clip_on=False)
    ax.plot([x1, x1], [y - 0.015 * ax.get_ylim()[1], y + 0.015 * ax.get_ylim()[1]], linewidth=1.2, clip_on=False)
    x_mid = (x0 + x1) / 2
    ax.text(x_mid, y, f"{label}: {value:.2f}", ha="center", va="bottom", fontsize=10, clip_on=False)

def plot_decomposition(m=300, pF0=2.0, pF1=4.0, pT=1.0, alpha=0.5, rho=0.5):
    F_A, T_A = marshallian_bundle(m, pF0, pT, alpha, rho)
    U_A = float(utility(F_A, T_A, alpha, rho))

    F_C, T_C = marshallian_bundle(m, pF1, pT, alpha, rho)
    U_C = float(utility(F_C, T_C, alpha, rho))

    F_upper_guess = max(m / min(pF0, pF1) * 1.4, F_A, F_C, 10)
    F_B, T_B, m_comp = hicksian_bundle(U_A, pF1, pT, alpha, rho, F_upper=F_upper_guess, n_grid=5000)

    x_max, y_max = auto_windows(m, pF0, pF1, pT, F_A, T_A, F_B, T_B, F_C, T_C)

    f = np.linspace(1e-3, x_max, 260)
    t = np.linspace(1e-3, y_max, 260)
    FF, TT = np.meshgrid(f, t)
    UU = utility(FF, TT, alpha, rho)

    fig, ax = plt.subplots(figsize=(9, 7))

    base_levels = [0.75 * U_A, U_A, 0.9 * U_C, U_C]
    levels = sorted(set(round(v, 4) for v in base_levels if v > 0))
    ax.contour(FF, TT, UU, levels=levels)

    T_old  = budget_line(f, m,      pF0, pT)
    T_new  = budget_line(f, m,      pF1, pT)
    T_comp = budget_line(f, m_comp, pF1, pT)

    ax.plot(f[(T_old  >= 0) & (T_old  <= y_max)], T_old [(T_old  >= 0) & (T_old  <= y_max)], linewidth=2,              label="Initial budget")
    ax.plot(f[(T_new  >= 0) & (T_new  <= y_max)], T_new [(T_new  >= 0) & (T_new  <= y_max)], linewidth=2, linestyle="--", label="New budget")
    ax.plot(f[(T_comp >= 0) & (T_comp <= y_max)], T_comp[(T_comp >= 0) & (T_comp <= y_max)], linewidth=2, linestyle=":",  label="Compensated budget")

    ax.scatter([F_A], [T_A], s=80, zorder=5, label="A: initial optimum")
    ax.scatter([F_B], [T_B], s=80, zorder=5, label="B: compensated optimum")
    ax.scatter([F_C], [T_C], s=80, zorder=5, label="C: final optimum")

    ax.annotate("A", (F_A, T_A), xytext=(5, 5), textcoords="offset points")
    ax.annotate("B", (F_B, T_B), xytext=(5, 5), textcoords="offset points")
    ax.annotate("C", (F_C, T_C), xytext=(5, 5), textcoords="offset points")

    for Fx, Tx, lbl in [(F_A, T_A, "A"), (F_B, T_B, "B"), (F_C, T_C, "C")]:
        ax.plot([Fx, Fx], [0, Tx], linestyle=":", linewidth=1)
        ax.text(Fx, -0.03 * y_max, lbl, ha="center", va="top", fontsize=10, clip_on=False)

    sub_F   = F_B - F_A
    inc_F   = F_C - F_B
    total_F = F_C - F_A

    effect_y1 = -0.10 * y_max
    effect_y2 = -0.18 * y_max
    effect_y3 = -0.26 * y_max

    draw_horizontal_effect(ax, F_A, F_B, effect_y1, "Substitution effect (Food)", sub_F)
    draw_horizontal_effect(ax, F_B, F_C, effect_y2, "Income effect (Food)",       inc_F)
    draw_horizontal_effect(ax, F_A, F_C, effect_y3, "Total effect (Food)",        total_F)

    ax.set_xlim(0, x_max)
    ax.set_ylim(effect_y3 - 0.08 * y_max, y_max)
    ax.set_xlabel("Food")
    ax.set_ylabel("T-shirts")
    ax.set_title("Substitution effect and income effect (food on the horizontal axis)")
    ax.legend(loc="best")
    plt.subplots_adjust(bottom=0.28)
    plt.show()

    sub_T   = T_B - T_A
    inc_T   = T_C - T_B
    total_T = T_C - T_A

    print(f"Income m = {m:.2f}")
    print(f"Initial price of Food: pF0 = {pF0:.2f}   |   New price of Food: pF1 = {pF1:.2f}   |   Price of T-shirts: pT = {pT:.2f}")
    print(f"Compensated income needed to stay on the initial utility: {m_comp:.2f}")
    print()
    print(f"A (initial optimum):     Food = {F_A:.2f},  T-shirts = {T_A:.2f},  U = {U_A:.2f}")
    print(f"B (compensated optimum): Food = {F_B:.2f},  T-shirts = {T_B:.2f},  U = {U_A:.2f}")
    print(f"C (final optimum):       Food = {F_C:.2f},  T-shirts = {T_C:.2f},  U = {U_C:.2f}")
    print()
    print("Decomposition for Food (horizontal axis)")
    print(f"  Substitution effect:  {sub_F:+.2f}")
    print(f"  Income effect:        {inc_F:+.2f}")
    print(f"  Total effect:         {total_F:+.2f}")
    print()
    print("Decomposition for T-shirts")
    print(f"  Substitution effect:  {sub_T:+.2f}")
    print(f"  Income effect:        {inc_T:+.2f}")
    print(f"  Total effect:         {total_T:+.2f}")
    print()
    if pF1 > pF0:
        print("Interpretation: the price of Food RISES. For normal goods, both the substitution")
        print("effect and the income effect push Food demand DOWN.")
    elif pF1 < pF0:
        print("Interpretation: the price of Food FALLS. For normal goods, both the substitution")
        print("effect and the income effect push Food demand UP.")
    else:
        print("No price change: the decomposition is trivial.")


In [ ]:
from ipywidgets import interact, FloatSlider

interact(
    plot_decomposition,
    m    =FloatSlider(value=300,  min=40,  max=600,  step=10,  description="Income m",  continuous_update=False),
    pF0  =FloatSlider(value=2.0,  min=0.5, max=8.0,  step=0.5, description="Old pF",    continuous_update=False),
    pF1  =FloatSlider(value=4.0,  min=0.5, max=8.0,  step=0.5, description="New pF",    continuous_update=False),
    pT   =FloatSlider(value=1.0,  min=0.5, max=4.0,  step=0.5, description="Price pT",  continuous_update=False),
    alpha=FloatSlider(value=0.5,  min=0.1, max=0.9,  step=0.05,description="alpha",     continuous_update=False),
    rho  =FloatSlider(value=0.5,  min=0.1, max=0.9,  step=0.05,description="rho",       continuous_update=False),
);
